# Universal Multi-Stock LSTM / Transformer (Full Pipeline)

Trains a single model on multiple symbols using the full feature set:
technical indicators + FinBERT sentiment embeddings + fundamental data.

**Prerequisites (run once before this notebook):**
- Market data: run the *Save stock price to cache* section of `runML.ipynb`
  with `timeframe = "1Day"` — this notebook requires daily bars.
  If hourly data is already cached it will be detected and replaced automatically.
- Fundamentals: fetched automatically below via yfinance (no API key needed).
- News articles: optional. Run `NewsPipeline` or `KaggleImporter` to populate
  `ArticleRepository`. Without articles the sentiment branch uses zero vectors
  (the model still trains, just without a sentiment signal).

**Flow:**
1. Ensure daily OHLCV bars are in `MarketDataCache`
2. Fetch fundamentals from yfinance if not already cached
3. Load or compute daily-aggregated sentiment (cached to `data/sentiment_daily.pkl`)
4. Build fused dataset per symbol (`build_dataset`), pool across symbols
5. Chronological train / val / test split + scaling (`make_loaders`)
6. Train `SentimentLSTM` or `SentimentTransformer`
7. Bootstrap evaluation (AUC + 95 % CI)
8. Live inference on a single stock

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
import torch
from pathlib import Path

from sentiment.log import setup_logging
from sentiment.sources.alpaca import AlpacaSource
from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalSource, FundamentalCache
from sentiment.embeddings import SentimentCache
from sentiment.features.technical import TechnicalFactors
from sentiment.features.dataloader import FusedDataset, build_dataset, make_loaders
from sentiment.model.lstm import SentimentLSTM
from sentiment.model.transformer import SentimentTransformer
from sentiment.model.train import train_model, bootstrap_evaluate

setup_logging()

C:\Users\dnjs2\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'SentimentCache' from 'sentiment.embeddings' (C:\TFA\ProjectCode\quant-sentiment-score\sentiment\embeddings\__init__.py)

## Config

In [ ]:
YEAR = 2018
WINDOW = 64
BATCH_SIZE = 32
N_EPOCHS = 100
MODEL_TYPE = "lstm"  # "lstm" or "transformer"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

symbols = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "AVGO", "PEP", "COST",
]

print(f"Device: {DEVICE}")

## Market Data

Technical indicators (SMA-60, RSI-14, ATR-14, …) require **daily** bars.
If the cache already holds daily data for all symbols, this cell skips the fetch.
If data is missing or is sub-daily (hourly), it re-fetches from Alpaca and
overwrites the cached file.

In [4]:
from datetime import datetime

cache = MarketDataCache()


def _is_daily(df: pd.DataFrame) -> bool:
    """Return True when the median gap between rows is ≥ 20 hours (daily cadence)."""
    if len(df) < 2:
        return False
    median_gap = df.index.to_series().diff().median()
    return median_gap >= pd.Timedelta("20h")


alpaca = AlpacaSource()
start = datetime(YEAR, 1, 1)
end   = datetime(YEAR, 12, 31)

for symbol in symbols:
    try:
        df = cache.load(symbol, YEAR)
    except FileNotFoundError:
        df = pd.DataFrame()
    
    if _is_daily(df):
        print(f"{symbol}: daily data already cached ({len(df)} bars)")
        continue

    reason = "sub-daily frequency" if df is not None and not df.empty else "not in cache"
    print(f"{symbol}: {reason} — fetching daily bars from Alpaca…")
    df = alpaca.fetch_bars(
        symbols=[symbol],
        timeframe="1Day",
        start=start,
        end=end,
    )
    cache.store(symbol, YEAR, df)
    print(f"  stored {len(df)} daily bars")

AAPL: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
MSFT: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
NVDA: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
AMZN: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
GOOGL: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
META: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
TSLA: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
AVGO: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
PEP: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars
COST: not in cache — fetching daily bars from Alpaca…
  stored 250 daily bars


## Fundamentals

Fetches 9 metrics (P/E, P/B, ROE, …) from yfinance and stores a dated snapshot
in `FundamentalCache`.  Skipped if a snapshot for each symbol already exists.

In [5]:
fund_cache  = FundamentalCache()
fund_source = FundamentalSource()

fund_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in symbols:
    existing = fund_cache.load_df(symbol)
    if existing is not None and not existing.empty:
        fund_dfs[symbol] = existing
        print(f"{symbol}: {len(existing)} snapshot(s) already cached")
        continue

    print(f"{symbol}: fetching fundamentals from yfinance…")
    data = fund_source.fetch(symbol)
    fund_cache.store(symbol, data)
    fund_dfs[symbol] = fund_cache.load_df(symbol)
    print(f"  stored snapshot: {data}")

AAPL: fetching fundamentals from yfinance…


/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


  stored snapshot: {'pe': 31.853165, 'forward_pe': 27.013708, 'pb': 41.953983, 'ps': 8.490454, 'roe': 1.5202099, 'op_margin': 0.35374, 'profit_margin': 0.27037, 'de': 102.63, 'beta': 1.116}
MSFT: fetching fundamentals from yfinance…


/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


  stored snapshot: {'pe': 23.31082, 'forward_pe': 19.765795, 'pb': 7.084291, 'ps': 9.069626, 'roe': 0.34390998, 'op_margin': 0.47094002, 'profit_margin': 0.39044, 'de': 31.539, 'beta': 1.108}
NVDA: fetching fundamentals from yfinance…


...

/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


  stored snapshot: {'pe': 62.16602, 'forward_pe': 17.915136, 'pb': 5.374614, 'ps': 22.100992, 'roe': 0.33370999, 'op_margin': 0.31765, 'profit_margin': 0.36571997, 'de': 166.032, 'beta': 1.257}
PEP: fetching fundamentals from yfinance…


/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


  stored snapshot: {'pe': 25.140234, 'forward_pe': 16.463934, 'pb': 10.087754, 'ps': 2.1922572, 'roe': 0.42848, 'op_margin': 0.14068, 'profit_margin': 0.08773, 'de': 258.081, 'beta': 0.375}
COST: fetching fundamentals from yfinance…
  stored snapshot: {'pe': 50.77268, 'forward_pe': 43.39907, 'pb': 13.465804, 'ps': 1.5099607, 'roe': 0.29650998, 'op_margin': 0.03744, 'profit_margin': 0.029860001, 'de': 25.976, 'beta': 0.993}


/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")


## Sentiment

Loads per-symbol sentiment from `data/sentiment/<SYMBOL>.parquet` (written by
`prepare_data.ipynb` phase 4).  Symbols without a cache entry receive zero
vectors — the model still trains on technical + fundamental features only.

In [ ]:
sentiment_cache = SentimentCache()

sentiment_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in symbols:
    if sentiment_cache.exists(symbol):
        sentiment_dfs[symbol] = sentiment_cache.load(symbol)
        print(f"{symbol}: loaded {len(sentiment_dfs[symbol])} days from cache")
    else:
        print(f"{symbol}: no sentiment cache — will use zero vectors")
        sentiment_dfs[symbol] = None

## Build Datasets

One `FusedDataset` per symbol (chronological order preserved), then pooled
by concatenation.  `build_dataset` raises `RuntimeError` if a symbol has
fewer than 126 daily bars (60 indicator warmup + window=64 + 2 target rows).

In [ ]:
technical = TechnicalFactors()

all_datasets: list[FusedDataset] = []
for symbol in symbols:
    df = cache.load(symbol, YEAR)
    ds = build_dataset(
        df,
        technical,
        sentiment_df=sentiment_dfs[symbol],
        ticker=symbol,
        window=WINDOW,
        fundamental_df=fund_dfs[symbol],
        include_momentum_slope=True,
    )
    all_datasets.append(ds)
    print(f"{symbol}: {len(ds['y'])} windows  label_mean={ds['y'].mean():.3f}")

pooled: FusedDataset = {
    "X_tech":            np.concatenate([d["X_tech"]            for d in all_datasets]),
    "X_sentiment":       np.concatenate([d["X_sentiment"]       for d in all_datasets]),
    "X_fundamental":     np.concatenate([d["X_fundamental"]     for d in all_datasets]),
    "X_sentiment_probs": np.concatenate([d["X_sentiment_probs"] for d in all_datasets]),
    "y":                 np.concatenate([d["y"]                 for d in all_datasets]),
    "dates":             np.concatenate([d["dates"]             for d in all_datasets]),
}

# Sort globally by date so make_loaders' positional split is chronologically correct
# across all symbols (without this, test set = trailing windows of the last symbols only)
sort_idx = np.argsort(pooled["dates"], kind="stable")
pooled = {k: v[sort_idx] for k, v in pooled.items()}

print(f"\nPooled: {len(pooled['y'])} windows total  label_mean={pooled['y'].mean():.3f}")
print(f"X_tech:            {pooled['X_tech'].shape}")
print(f"X_sentiment:       {pooled['X_sentiment'].shape}")
print(f"X_sentiment_probs: {pooled['X_sentiment_probs'].shape}")
print(f"X_fundamental:     {pooled['X_fundamental'].shape}")

## DataLoaders

In [9]:
train_loader, val_loader, test_loader, tech_scaler, fund_scaler = make_loaders(
    pooled,
    test_frac=0.2,
    val_frac=0.1,
    batch_size=BATCH_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

10:00:18 INFO     sentiment.features.dataloader  Split sizes — train: 907, val: 101, test: 252


Train batches: 28
Val   batches: 4
Test  batches: 8


## Model

`n_fundamentals` and `n_sentiment_probs` are read from the pooled arrays so
the architecture automatically matches the available features — if fundamentals
or sentiment were missing, the corresponding model branch is disabled.

In [10]:
n_fundamentals    = pooled["X_fundamental"].shape[1]
n_sentiment_probs = pooled["X_sentiment_probs"].shape[2]

if MODEL_TYPE == "lstm":
    model = SentimentLSTM(
        n_factors=16,
        sentiment_dim=768,
        hidden_size=32,
        num_layers=2,
        dropout=0.2,
        n_fundamentals=n_fundamentals,
        n_sentiment_probs=n_sentiment_probs,
    ).to(DEVICE)
else:
    model = SentimentTransformer(
        n_factors=16,
        sentiment_dim=768,
        d_model=64,
        nhead=4,
        n_layers=6,
        dim_feedforward=128,
        dropout=0.2,
        n_fundamentals=n_fundamentals,
        n_sentiment_probs=n_sentiment_probs,
    ).to(DEVICE)

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

SentimentLSTM(
  (sentiment_proj): Linear(in_features=768, out_features=16, bias=True)
  (lstm): LSTM(35, 32, num_layers=2, batch_first=True, dropout=0.2)
  (classifier): Sequential(
    (0): Linear(in_features=42, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Parameters: 31,057


## Training

In [11]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    patience=15,
    device=DEVICE,
)

print(f"Best epoch: {history['best_epoch']}  |  Best val AUC: {history['best_val_auc']:.4f}")

10:00:32 INFO     sentiment.model.train  Epoch   1 | train_loss=0.7057 | val_loss=0.6998 | val_auc=0.4187 | val_acc=0.3861
10:00:32 INFO     sentiment.model.train  Epoch   2 | train_loss=0.6902 | val_loss=0.6979 | val_auc=0.4940 | val_acc=0.4455
10:00:33 INFO     sentiment.model.train  Epoch   3 | train_loss=0.6788 | val_loss=0.7056 | val_auc=0.4649 | val_acc=0.4752
10:00:33 INFO     sentiment.model.train  Epoch   4 | train_loss=0.6621 | val_loss=0.6890 | val_auc=0.5522 | val_acc=0.5545
10:00:33 INFO     sentiment.model.train  Epoch   5 | train_loss=0.6663 | val_loss=0.7038 | val_auc=0.5327 | val_acc=0.5050
10:00:33 INFO     sentiment.model.train  Epoch   6 | train_loss=0.6316 | val_loss=0.6903 | val_auc=0.5766 | val_acc=0.5050
10:00:34 INFO     sentiment.model.train  Epoch   7 | train_loss=0.6129 | val_loss=0.6988 | val_auc=0.5718 | val_acc=0.5545
10:00:34 INFO     sentiment.model.train  Epoch   8 | train_loss=0.6059 | val_loss=0.7169 | val_auc=0.5355 | val_acc=0.5644
10:00:34 INFO   

Best epoch: 13  |  Best val AUC: 0.7069


## Evaluation

In [12]:
result = bootstrap_evaluate(model, test_loader, device=DEVICE, n_bootstrap=1000, seed=42)

print(f"AUC      : {result['auc_mean']:.3f}  [{result['auc_ci_low']:.3f}, {result['auc_ci_high']:.3f}]")
print(f"Accuracy : {result['accuracy_mean']:.3f}  [{result['accuracy_ci_low']:.3f}, {result['accuracy_ci_high']:.3f}]")
print(f"Precision: {result['precision_mean']:.3f}  [{result['precision_ci_low']:.3f}, {result['precision_ci_high']:.3f}]")
print(f"Recall   : {result['recall_mean']:.3f}  [{result['recall_ci_low']:.3f}, {result['recall_ci_high']:.3f}]")

AUC      : 0.711  [0.644, 0.772]
Accuracy : 0.632  [0.571, 0.690]
Precision: 0.591  [0.500, 0.679]
Recall   : 0.649  [0.564, 0.734]


## Save Model

In [11]:
torch.save(model.state_dict(), f"universal_{MODEL_TYPE}.pt")
print(f"Saved to universal_{MODEL_TYPE}.pt")

NameError: name 'model' is not defined

## Live Inference

Take the last 64-day window for a single stock and predict
P(close[t+2] > close[t-1]).

In [ ]:
infer_symbol = "AAPL"

df_infer = cache.load(infer_symbol, YEAR)
ds_infer = build_dataset(
    df_infer,
    technical,
    sentiment_df=sentiment_df,
    ticker=infer_symbol,
    window=WINDOW,
    fundamental_df=fund_dfs[infer_symbol],
    include_momentum_slope=True,
)

# Take the last window and apply the fitted scalers
X_tech_last  = ds_infer["X_tech"][[-1]].copy()           # (1, window, 16)
X_sent_last  = ds_infer["X_sentiment"][[-1]]              # (1, window, 768)
X_fund_last  = ds_infer["X_fundamental"][[-1]].copy()     # (1, n_fund)
X_sprob_last = ds_infer["X_sentiment_probs"][[-1]]        # (1, window, 3)

n, w, f = X_tech_last.shape
X_tech_last = tech_scaler.transform(X_tech_last.reshape(-1, f)).reshape(n, w, f).astype(np.float32)
if fund_scaler is not None:
    X_fund_last = fund_scaler.transform(X_fund_last).astype(np.float32)

model.eval()
with torch.no_grad():
    logit = model(
        torch.tensor(X_tech_last).to(DEVICE),
        torch.tensor(X_sent_last).to(DEVICE),
        fundamentals=torch.tensor(X_fund_last).to(DEVICE),
        sentiment_probs=torch.tensor(X_sprob_last).to(DEVICE),
    )
    prob_up = torch.sigmoid(logit).item()

last_close = df_infer["close"].iloc[-1]
print(f"{infer_symbol} last close  : ${last_close:.2f}")
print(f"P(price goes up) : {prob_up:.4f}")
print(f"Signal           : {'BUY' if prob_up > 0.5 else 'HOLD/SELL'}")